# Zn control with iodine-bearing materials withheld

Retrain the standard nine-element encoder and the three Zn heads from scratch on the canonical Figshare splits after removing every material with a canonical Zn spectrum and iodine in its structure. The heads are ExpertXAS (310–500–750), UniversalXAS, and Zn-tuned UniversalXAS. Use the previous experiment's model settings and validation criteria.

The surviving materials retain their original splits. The removed Zn spectra form a separate holdout, including sites that previously appeared in training. The holdout is specific to this control. The historical models may already have seen some of these materials.

Run in the OmniXAS Python 3.11 training environment. Set RUN_TRAINING to True to launch the encoder and head sweeps. This one-off keeps weights and features in memory and shows results inline.

In [ ]:
from collections import Counter
from pathlib import Path
from unittest.mock import patch
import hashlib
import json
import os
import random
import sys

import matplotlib.pyplot as plt
import numpy as np
import pandas as pd
from IPython.display import display

REPO_ROOT = next(p for p in (Path.cwd(), *Path.cwd().parents) if (p / "omnixas").is_dir())
sys.path.insert(0, str(REPO_ROOT))
sys.path.insert(0, str(REPO_ROOT / "tutorial_omnixas"))
os.environ.setdefault("DGLBACKEND", "pytorch")
sys.dont_write_bytecode = True

DATA_ROOT = Path(os.environ.get("OMNIXAS_DATA_ROOT", REPO_ROOT.parent / "OmniXAS_data")).expanduser().resolve()
DATASET_ROOT = DATA_ROOT / "figshare_141_splits_unscaled_zn9"
RAW_ROOT = DATA_ROOT / "figshare"
ELEMENTS = ("Ti", "V", "Cr", "Mn", "Fe", "Co", "Ni", "Cu", "Zn")
TASKS = [f"{element}_FEFF" for element in ELEMENTS]
SPLITS = ("train", "val", "test")
SPLIT_SEED = 42
ENCODER_SEED = 123
RUN_TRAINING = False

EXPERT_WIDTHS = (310, 500, 750)
UNIVERSAL_WIDTHS = (500, 500, 550)
EXPERT_HP = {"lr": 0.002, "dropout": 0.05, "schedule": "plateau", "seed": 43}
UNIVERSAL_TRIALS, TUNED_TRIALS = 100, 300
ENCODER_EPOCHS, EXPERT_EPOCHS = 1000, 800
UNIVERSAL_EPOCHS, TUNED_EPOCHS = 800, 1000
BATCH_SIZE, PATIENCE = 4096, 60
ENERGY_GRID = np.linspace(9659.0, 9694.0, 141)
SIGMA_EV = 0.5

def sha256(path):
    with path.open("rb") as handle:
        return hashlib.file_digest(handle, "sha256").hexdigest()

def split_id(value):
    material, site = value.rsplit("_", 1)
    return material, int(site)

## Audit and filter the canonical dataset

Inspect the archived structures for every canonical Zn site. If any Zn site belongs to a structure containing iodine, exclude that material across all nine absorbers and all three canonical splits. Validate source-site order and report the removed rows with their original split.

In [ ]:
dataset_file = DATASET_ROOT / "dataset.json"
dataset_hash = sha256(dataset_file)
dataset_info = json.loads(dataset_file.read_text(encoding="utf-8"))
if dataset_info.get("target_scale") != 1 or dataset_info.get("feature_scale") != 1000:
    raise ValueError("Expected the previous unscaled nine-element dataset and 1000× feature scale.")
if not np.array_equal(dataset_info.get("zn_grid_ev"), ENERGY_GRID):
    raise ValueError("The saved Zn grid differs from the previous experiment.")

canonical = {}
key_split, material_split = {}, {}
for element in ELEMENTS:
    for split in SPLITS:
        prefix = f"{element}_FEFF_{split}"
        ids = (DATASET_ROOT / "material_id_and_site" / f"{prefix}.txt").read_text(encoding="utf-8-sig").splitlines()
        y = np.loadtxt(DATASET_ROOT / "ml_data" / f"{prefix}_y.txt", dtype=np.float32, ndmin=2)
        if not ids or y.shape != (len(ids), 141) or not np.isfinite(y).all() or (y < 0).any():
            raise ValueError(f"Invalid canonical IDs or targets for {prefix}.")
        keys = [(element, *split_id(value)) for value in ids]
        for key in keys:
            _, material, site = key
            if not material or site < 0 or key in key_split:
                raise ValueError(f"Invalid or duplicate canonical key {key}.")
            key_split[key] = split
            if material_split.setdefault(material, split) != split:
                raise ValueError(f"Material appears in multiple canonical splits: {material}")
        canonical[element, split] = (ids, keys, y)

withheld_materials = set()
for split in SPLITS:
    for element, material, site in canonical["Zn", split][1]:
        path = RAW_ROOT / "FEFF" / element / material / "FEFF-XANES" / f"{site:03d}_{element}" / "structure.json"
        atoms = json.loads(path.read_bytes())["sites"]
        if site >= len(atoms) or not any(atom["element"] == "Zn" and atom.get("occu", 1) > 0
                                          for atom in atoms[site]["species"]):
            raise ValueError(f"Archived absorber index disagrees with the Zn row: {path}")
        if any(atom["element"] == "I" and atom.get("occu", 1) > 0 for entry in atoms for atom in entry["species"]):
            withheld_materials.add(material)
if not withheld_materials:
    raise ValueError("The canonical Zn splits contain no iodine-bearing materials.")

base, holdout, removed = {}, {}, []
for (element, split), (ids, keys, y) in canonical.items():
    keep = np.array([key[1] not in withheld_materials for key in keys], dtype=bool)
    if not keep.any():
        raise ValueError(f"Filtering emptied {element} {split}.")
    base[element, split] = ([ids[i] for i in np.flatnonzero(keep)], [keys[i] for i in np.flatnonzero(keep)], y[keep])
    for i in np.flatnonzero(~keep):
        _, material, site = keys[i]
        removed.append({"absorber": element, "material": material, "site": site, "original split": split})
    if element == "Zn" and not keep.all():
        holdout[split] = ([keys[i] for i in np.flatnonzero(~keep)], y[~keep])

assert withheld_materials.isdisjoint(key[1] for _, keys, _ in base.values() for key in keys)
print(f"Source dataset SHA-256: {dataset_hash}")
print(f"Withheld {len(withheld_materials)} Zn materials and {len(removed)} site rows across all tasks.")
display(pd.DataFrame(removed).sort_values(["absorber", "material", "site"]))

## Fit a fresh nine-element encoder

Use the previous balanced batches, seed, spectral-difference loss, and encoder validation criterion on the filtered training and validation rows. Freeze the selected encoder and export features in memory. Encode the withheld structures after head selection.

In [ ]:
if not RUN_TRAINING:
    raise RuntimeError("The input audit is complete. Set RUN_TRAINING = True and rerun to train.")

from io import BytesIO
from tempfile import TemporaryDirectory

import lightning.pytorch as pl
from lightning.pytorch.callbacks import EarlyStopping
from torch.utils.data import DataLoader
from tqdm.std import tqdm
import omnixas.data.feff_graph as graph_data
from omnixas.data.feff_graph import (
    BalancedTaskBatchSampler, CollateGraphs, FEFFDataset,
    patch_matgl_gpu_constants,
)
from omnixas.model.m3gnet_xas import M3GNetXAS
from omnixas.model.xasblock import XASBlock
import torch

device = torch.device("cuda" if torch.cuda.is_available() else "cpu")
FEFF_TASKS = TASKS
ENCODER_MONITOR = "val_balanced_rel_mse"

def snapshot(model):
    return {key: value.detach().cpu().clone() for key, value in model.state_dict().items()}

def state_sha256(state):
    buffer = BytesIO()
    torch.save(state, buffer)
    return hashlib.sha256(buffer.getbuffer()).hexdigest()

def encode_batch(model, batch, expected_rows):
    with torch.inference_mode():
        X = model.encode(batch["graph"].to(device), batch["line_graph"].to(device),
                         batch["site"].to(device), scaled=True).cpu().numpy()
    if X.shape != (expected_rows, 64) or not np.isfinite(X).all():
        raise ValueError("Invalid encoded feature batch.")
    return X

class LitScratch(pl.LightningModule):
    def __init__(self, model: M3GNetXAS, train_base: torch.Tensor, val_base: torch.Tensor, lr: float):
        super().__init__()
        self.model, self.lr = model, lr
        self.register_buffer("train_base", train_base)
        self.register_buffer("val_base", val_base)
        self.val_mse, self.val_task = [], []

    def step(self, batch, stage):
        graph = batch["graph"].to(self.device)
        line_graph = batch["line_graph"].to(self.device)
        site = batch["site"].to(self.device)
        pred = self.model(graph, line_graph, site)
        y = batch["y"].to(self.device)
        task = batch["task"].to(self.device)
        mse = ((pred - y) ** 2).mean(1)
        base = self.train_base if stage == "train" else self.val_base
        loss = (mse / base[task].clamp_min(1e-12)).mean() + 0.02 * (
            torch.diff(pred, dim=1) - torch.diff(y, dim=1)
        ).square().mean()
        self.log(f"{stage}_loss", loss, on_step=False, on_epoch=True, prog_bar=True, batch_size=y.shape[0])
        if stage == "val":
            self.val_mse.append(mse.detach())
            self.val_task.append(task.detach())
        return loss

    def training_step(self, batch, _):
        return self.step(batch, "train")

    def on_validation_epoch_start(self):
        self.val_mse, self.val_task = [], []

    def validation_step(self, batch, _):
        return self.step(batch, "val")

    def on_validation_epoch_end(self):
        if self.trainer.sanity_checking or not self.val_mse:
            return
        mses, tasks = torch.cat(self.val_mse), torch.cat(self.val_task)
        rel = [
            mses[tasks == i].median() / self.val_base[i].clamp_min(1e-12)
            for i in range(len(FEFF_TASKS))
            if (tasks == i).any()
        ]
        if len(rel) != len(FEFF_TASKS):
            raise RuntimeError("Validation lacks one or more FEFF elements")
        self.log("val_balanced_rel_mse", torch.stack(rel).mean(), on_epoch=True, prog_bar=True)

    def configure_optimizers(self):
        opt = torch.optim.AdamW(
            self.parameters(),
            lr=self.lr,
            weight_decay=1e-5,
            fused=self.device.type == "cuda",
        )
        sched = torch.optim.lr_scheduler.ReduceLROnPlateau(opt, mode="min", factor=0.5, patience=16, min_lr=1e-6)
        return {
            "optimizer": opt,
            "lr_scheduler": {"scheduler": sched, "monitor": "val_balanced_rel_mse"},
        }

class BestEncoderInMemory(pl.Callback):
    def __init__(self):
        self.metric, self.epoch, self.state = float("inf"), None, None

    def on_validation_end(self, trainer, module):
        if trainer.sanity_checking:
            return
        value = float(trainer.callback_metrics[ENCODER_MONITOR])
        if not np.isfinite(value):
            raise ValueError("Non-finite encoder validation metric.")
        if trainer.current_epoch % 10 == 0:
            print(f"Encoder epoch {trainer.current_epoch}: validation macro relative median MSE {value:.6g}", flush=True)
        if value < self.metric:
            self.metric, self.epoch = value, trainer.current_epoch
            self.state = snapshot(module.model)

def filtered_dataset(tasks, split):
    dataset = FEFFDataset(REPO_ROOT, RAW_ROOT, tasks, split, DATASET_ROOT)
    dataset.rows = [row for row in dataset.rows if row[1] not in withheld_materials]
    expected = [(task, material, site) for task in tasks
                for _, material, site in base[task.split("_", 1)[0], split][1]]
    if [row[:3] for row in dataset.rows] != expected:
        raise ValueError(f"Filtered graph rows disagree with the canonical data for {split}: {tasks}")
    return dataset

def fit_encoder():
    patch_matgl_gpu_constants()
    pl.seed_everything(ENCODER_SEED, workers=True)
    torch.set_float32_matmul_precision("high")
    model = M3GNetXAS(head_dropout=0.05)
    bases = [torch.tensor([np.median(
        ((base[element, split][2] - base[element, "train"][2].mean(0)) ** 2).mean(1)
    ) for element in ELEMENTS]) for split in ("train", "val")]
    best = BestEncoderInMemory()

    with patch.object(graph_data, "FEFF_TASKS", TASKS), TemporaryDirectory(prefix="omnixas-zn-withheld-") as scratch:
        collator = CollateGraphs(model.encoder)
        train = filtered_dataset(TASKS, "train")
        val = filtered_dataset(TASKS, "val")
        train_counts = Counter(row[0] for row in train.rows)
        sampler = BalancedTaskBatchSampler(train.rows, min(64, min(train_counts.values())), ENCODER_SEED)
        train_loader = DataLoader(train, batch_sampler=sampler, collate_fn=collator, num_workers=0)
        val_loader = DataLoader(val, batch_size=1024, collate_fn=collator, num_workers=0)
        trainer = pl.Trainer(
            max_epochs=ENCODER_EPOCHS, accelerator="auto", devices=1,
            precision="bf16-mixed" if device.type == "cuda" and torch.cuda.is_bf16_supported() else "32-true",
            callbacks=[best, EarlyStopping(monitor=ENCODER_MONITOR, patience=60, mode="min")],
            logger=False, enable_progress_bar=False, enable_checkpointing=False, default_root_dir=scratch,
        )
        trainer.fit(LitScratch(model, *bases, 0.001), train_loader, val_loader)

    if best.state is None:
        raise RuntimeError("No encoder validation checkpoint was selected.")
    model.load_state_dict(best.state, strict=True)
    model.to(device).eval().requires_grad_(False)
    return model, state_sha256(best.state), {"seed": ENCODER_SEED, "epoch": best.epoch, "validation": best.metric}


def ordered_features(model, collator, dataset, keys, reference_y):
    source = [(task.split("_", 1)[0], material, site) for task, material, site, _ in dataset.rows]
    if source != keys:
        raise ValueError("Encoded rows do not match the requested IDs.")
    loader = DataLoader(dataset, batch_size=1024, collate_fn=collator, num_workers=0)
    xs, ys = [], []
    for batch in loader:
        xs.append(encode_batch(model, batch, len(batch["y"])))
        ys.append(batch["y"].numpy())
    X, y = np.concatenate(xs), np.concatenate(ys)
    if X.shape != (len(keys), 64) or not np.array_equal(y, reference_y):
        raise ValueError("Encoded rows do not align with target rows.")
    collator._graph_cache.clear()
    return X, y

def export_base_features(model):
    data = {}
    with patch.object(graph_data, "FEFF_TASKS", TASKS):
        collator = CollateGraphs(model.encoder)
    for (element, split), (_, keys, y) in tqdm(base.items(), desc="Encode filtered nine-element splits"):
        dataset = filtered_dataset([f"{element}_FEFF"], split)
        data[element, split] = ordered_features(model, collator, dataset, keys, y)
    return data

encoder_model, encoder_hash, encoder_summary = fit_encoder()
print(f"Fresh encoder state SHA-256: {encoder_hash}")
display(pd.DataFrame([encoder_summary]))
base_features = export_base_features(encoder_model)

## Fit the three Zn heads

Train the fixed 310–500–750 ExpertXAS on filtered Zn train rows. Train UniversalXAS on the balanced nine-element training data, then tune the selected head on filtered Zn train rows. Use the original sweep sizes and select heads by filtered Zn validation median MSE.

In [ ]:
def make_head(widths, dropout, state=None):
    with patch.object(XASBlock, "DROPOUT", dropout):
        model = XASBlock(64, list(widths), 141).to(device)
    if state is not None:
        model.load_state_dict(state, strict=True)
    return model

def predict_head(model, X):
    model.eval()
    with torch.inference_mode():
        predictions = torch.cat([model(batch.to(device)).cpu()
                                 for batch in torch.from_numpy(X).split(BATCH_SIZE)]).numpy()
    if predictions.shape != (len(X), 141) or not np.isfinite(predictions).all():
        raise ValueError("Invalid prediction batch.")
    return predictions

def training_baseline(y):
    return float(np.median(((y - y.mean(axis=0)) ** 2).mean(axis=1)))

rng = np.random.default_rng(SPLIT_SEED)
n_per_task = max(len(base_features[element, "train"][0]) for element in ELEMENTS)
balanced_X, balanced_y, balanced_tasks, loss_baselines = [], [], [], []
for task_id, element in enumerate(ELEMENTS):
    X, y = base_features[element, "train"]
    indices = np.r_[np.arange(len(X)), rng.integers(len(X), size=n_per_task - len(X))]
    rng.shuffle(indices)
    balanced_X.append(X[indices])
    balanced_y.append(y[indices])
    balanced_tasks.append(np.full(len(indices), task_id, dtype=np.int64))
    loss_baselines.append(training_baseline(y))

universal_X = torch.as_tensor(np.concatenate(balanced_X), device=device)
universal_y = torch.as_tensor(np.concatenate(balanced_y), device=device)
universal_groups = torch.as_tensor(np.concatenate(balanced_tasks), device=device)
group_baselines = torch.as_tensor(loss_baselines, dtype=torch.float32, device=device).clamp_min(1e-12)
zn_train_X, zn_train_y = (torch.as_tensor(arr, device=device) for arr in base_features["Zn", "train"])
zn_val_X, zn_val_y = (torch.as_tensor(arr, device=device) for arr in base_features["Zn", "val"])

def fit_head(X, y, widths, hp, epochs, *, source=None, groups=None):
    random.seed(hp["seed"])
    np.random.seed(hp["seed"])
    torch.manual_seed(hp["seed"])
    if device.type == "cuda":
        torch.cuda.manual_seed_all(hp["seed"])
    if len(X) < 2:
        raise ValueError("BatchNorm requires at least two training examples.")
    model = make_head(widths, hp["dropout"], source)
    optimizer = torch.optim.Adam(model.parameters(), lr=hp["lr"], fused=device.type == "cuda")
    plateau = hp["schedule"] == "plateau"
    scheduler = (torch.optim.lr_scheduler.ReduceLROnPlateau(optimizer, mode="min", factor=0.5, patience=16, min_lr=1e-6)
                 if plateau else torch.optim.lr_scheduler.CosineAnnealingLR(optimizer, T_max=500, eta_min=1e-6))

    def validation_median():
        model.eval()
        with torch.inference_mode():
            median = float(torch.quantile((model(zn_val_X) - zn_val_y).square().mean(1), 0.5))
        if not np.isfinite(median):
            raise ValueError("Non-finite Zn validation median MSE.")
        return median

    best, best_epoch, best_state = float("inf"), None, None
    if source is not None:
        best, best_state = validation_median(), snapshot(model)
        if plateau:
            scheduler.step(best)
    stale = 0
    for epoch in range(1, epochs + 1):
        model.train()
        batches = list(torch.randperm(len(X), device=device).split(BATCH_SIZE))
        if len(batches) > 1 and len(batches[-1]) == 1:
            batches[-2:] = [torch.cat(batches[-2:])]
        for index in batches:
            optimizer.zero_grad(set_to_none=True)
            errors = (model(X[index]) - y[index]).square().mean(1)
            if groups is None:
                loss = errors.mean()
            else:
                group = groups[index]
                loss = torch.stack([errors[group == i].mean() / group_baselines[i].pow(1.5)
                                    for i in torch.unique(group)]).mean()
            loss.backward()
            optimizer.step()
        metric = validation_median()
        if metric < best:
            best, best_epoch, best_state = metric, epoch, snapshot(model)
            stale = 0
        else:
            stale += 1
        if plateau:
            scheduler.step(metric)
        else:
            scheduler.step()
        if stale >= PATIENCE:
            break
    if best_state is None:
        raise RuntimeError("No head validation checkpoint was selected.")
    return best_state, {**hp, "Zn validation median MSE": best, "best epoch": best_epoch, "epochs run": epoch}

def trial_settings(count, seed, max_lr):
    rng = np.random.default_rng(seed)
    return [{"lr": float(np.exp(rng.uniform(np.log(1e-5), np.log(max_lr)))),
             "dropout": float(rng.choice([0.0, 0.05, 0.10, 0.20])),
             "schedule": str(rng.choice(["plateau", "cosine"])),
             "seed": int(rng.integers(1_000_000))} for _ in range(count)]

def sweep(label, X, y, settings, epochs, *, source=None, groups=None):
    winner, results = None, []
    progress = tqdm(settings, desc=label)
    for trial, hp in enumerate(progress, 1):
        state, record = fit_head(X, y, UNIVERSAL_WIDTHS, hp, epochs, source=source, groups=groups)
        record = {"trial": trial, **record}
        results.append(record)
        if winner is None or record["Zn validation median MSE"] < winner[1]["Zn validation median MSE"]:
            winner = state, record
        progress.set_postfix(best=winner[1]["Zn validation median MSE"])
    return winner, pd.DataFrame(results)

In [ ]:
expert_state, expert_config = fit_head(zn_train_X, zn_train_y, EXPERT_WIDTHS, EXPERT_HP, EXPERT_EPOCHS)
print("ExpertXAS fit complete.", flush=True)
universal_settings = trial_settings(UNIVERSAL_TRIALS, 1042, 1e-2)
(universal_state, universal_config), universal_trials = sweep(
    "UniversalXAS", universal_X, universal_y, universal_settings,
    UNIVERSAL_EPOCHS, groups=universal_groups,
)
(tuned_state, tuned_config), tuned_trials = sweep(
    "Tuned-UniversalXAS", zn_train_X, zn_train_y, trial_settings(TUNED_TRIALS, 2042, 5e-3),
    TUNED_EPOCHS, source=universal_state,
)

model_heads = {
    "ExpertXAS": make_head(EXPERT_WIDTHS, expert_config["dropout"], expert_state),
    "UniversalXAS": make_head(UNIVERSAL_WIDTHS, universal_config["dropout"], universal_state),
    "Tuned-UniversalXAS": make_head(UNIVERSAL_WIDTHS, tuned_config["dropout"], tuned_state),
}
display(pd.DataFrame([expert_config, universal_config, tuned_config], index=model_heads))
print(f"UniversalXAS initialization for tuning: {state_sha256(universal_state)}")
if tuned_config["best epoch"] is None:
    print("The selected tuned head retains the UniversalXAS weights.")

## Evaluate the remaining Zn test spectra

Report mean MSE, median MSE, and eta. Use the filtered Zn train mean for every head. The changed Zn baseline makes these eta values specific to this control. Compare raw MSE only on identical material and site IDs when comparing to a previous run.

In [ ]:
from scipy.stats import pearsonr, spearmanr

test_ids, test_keys, test_y = base["Zn", "test"]
test_X, _ = base_features["Zn", "test"]
train_mean = base["Zn", "train"][2].mean(axis=0)
model_predictions = {name: predict_head(model, test_X) for name, model in model_heads.items()}
model_errors = {name: ((prediction - test_y) ** 2).mean(axis=1)
                for name, prediction in model_predictions.items()}
errors = model_errors["ExpertXAS"]
baseline_errors = ((test_y - train_mean) ** 2).mean(axis=1)

def error_ratio(baseline, model):
    return float(baseline / model) if model > 0 else (float("inf") if baseline > 0 else float("nan"))

comparison = pd.DataFrame([
    {"model": name, "head parameters": sum(p.numel() for p in model_heads[name].parameters()),
     "test MSE": values.mean(), "test median MSE": np.median(values),
     "test eta": error_ratio(np.median(baseline_errors), np.median(values))}
    for name, values in model_errors.items()
])
display(comparison)

def noise_ratio(y):
    scale = float(np.std(y))
    if scale == 0:
        return 0.0
    sigma = SIGMA_EV / (ENERGY_GRID[1] - ENERGY_GRID[0])
    half = max(1, int(3 * sigma))
    x = np.arange(-half, half + 1)
    kernel = np.exp(-0.5 * (x / sigma) ** 2)
    kernel /= kernel.sum()
    smooth = np.convolve(np.pad(y, half, mode="edge"), kernel, mode="valid")
    return float(np.std(y - smooth) / scale)

ratios = np.array([noise_ratio(y) for y in test_y])
epsilon = np.finfo(float).eps
pooled = np.concatenate(list(model_errors.values()))
xmax, ymax = max(float(np.percentile(pooled, 99)), epsilon), max(float(np.percentile(ratios, 99)), epsilon)
names = list(model_errors)

fig, axes = plt.subplots(2, len(names), figsize=(5.1 * len(names), 8), sharey="row", squeeze=False)
for column, name in enumerate(names):
    values = model_errors[name]
    for row, ax in enumerate(axes[:, column]):
        ax.scatter(values, ratios, s=14, alpha=0.5)
        ax.set_xlabel("Per-spectrum test MSE")
        ax.grid(alpha=0.2)
        if row == 0:
            ax.set_xlim(0, max(float(pooled.max()) * 1.05, epsilon))
        else:
            ax.set_xlim(0, xmax)
            ax.set_ylim(0, ymax)
    if np.ptp(values) and np.ptp(ratios):
        pearson, spearman = pearsonr(values, ratios).statistic, spearmanr(values, ratios).statistic
        axes[0, column].set_title(f"{name} | Pearson {pearson:.2f} | Spearman {spearman:.2f}")
    else:
        axes[0, column].set_title(name)
axes[0, 0].set_ylabel("Detail ratio, all test rows")
axes[1, 0].set_ylabel("Detail ratio, pooled 99th percentile zoom")
fig.suptitle("Filtered Zn test error and spectral detail")
fig.tight_layout()
display(fig)
plt.close(fig)

def plot_spectra(ax, grid, y, predictions, title, xlabel="Energy (eV)"):
    ax.plot(grid, y, color="black", linewidth=1.5, label="Target")
    for name, prediction in predictions.items():
        ax.plot(grid, prediction, label=name)
    ax.set_title(title, fontsize=9)
    ax.set_xlabel(xlabel)
    ax.set_ylabel("Intensity (unscaled)")
    ax.grid(alpha=0.2)

In [ ]:
def plot_grid(panels, title, ncol=3):
    nrow = (len(panels) + ncol - 1) // ncol
    fig, axes = plt.subplots(nrow, ncol, figsize=(4.8 * ncol, 3.35 * nrow), squeeze=False)
    for ax, panel in zip(axes.flat, panels):
        plot_spectra(ax, *panel)
    for ax in list(axes.flat)[len(panels):]:
        ax.set_visible(False)
    axes.flat[0].legend(fontsize=7)
    fig.suptitle(title)
    fig.tight_layout()
    display(fig)
    plt.close(fig)

def test_panel(index, label):
    _, material, site = test_keys[index]
    return (ENERGY_GRID, test_y[index], {name: model_predictions[name][index] for name in names},
            f"Zn {material} site {site:03d} | {label}\nExpertXAS MSE {errors[index]:.4g}")

percentiles = (5, 10, 20, 30, 50, 70, 80, 90, 95)
available = list(range(len(errors)))
panels = []
for p in percentiles:
    if not available:
        break
    index = min(available, key=lambda i: abs(errors[i] - np.percentile(errors, p)))
    available.remove(index)
    panels.append(test_panel(index, f"near P{p}"))
plot_grid(panels, "Zn test spectra by ExpertXAS MSE percentile")

p95 = np.percentile(errors, 95)
seen_materials, panels = set(), []
for index in np.argsort(abs(errors - p95), kind="stable"):
    material = test_keys[index][1]
    if material in seen_materials:
        continue
    seen_materials.add(material)
    panels.append(test_panel(index, "near P95"))
    if len(panels) == 9:
        break
plot_grid(panels, "Zn test spectra near P95 | distinct materials")

## Evaluate the withheld Zn–I materials

Score every withheld Zn spectrum using the selected heads. Show its original split and report the original Zn test subset separately. The train-mean baseline uses only the filtered Zn training targets.

In [ ]:
with patch.object(graph_data, "FEFF_TASKS", TASKS):
    holdout_collator = CollateGraphs(encoder_model.encoder)

holdout_rows, panels = [], []
for split in ("test", "val", "train"):
    if split not in holdout:
        continue
    keys, y = holdout[split]
    dataset = FEFFDataset(REPO_ROOT, RAW_ROOT, ["Zn_FEFF"], split, DATASET_ROOT)
    dataset.rows = [row for row in dataset.rows if row[1] in withheld_materials]
    X, y = ordered_features(encoder_model, holdout_collator, dataset, keys, y)
    predictions = {name: predict_head(model, X) for name, model in model_heads.items()}
    baseline = ((y - train_mean) ** 2).mean(axis=1)
    for index, (_, mid, site) in enumerate(keys):
        curves = {}
        for name, predicted in predictions.items():
            mse = float(np.mean((predicted[index] - y[index]) ** 2))
            eta = error_ratio(baseline[index], mse)
            holdout_rows.append({"material": mid, "site": site, "original split": split, "model": name,
                                 "site MSE": mse, "baseline MSE": float(baseline[index]), "single-spectrum eta": eta})
            curves[f"{name} ({eta:.2f}×)"] = predicted[index]
        panels.append((ENERGY_GRID, y[index], curves, f"Zn {mid} site {site:03d} | originally {split}"))

holdout_scores = pd.DataFrame(holdout_rows)
group_scores = []
for label, subset in (("All withheld Zn–I", holdout_scores),
                      ("Original Zn test Zn–I", holdout_scores[holdout_scores["original split"] == "test"])):
    for name, group in subset.groupby("model", sort=False):
        median = group["site MSE"].median()
        group_scores.append({"subset": label, "model": name, "sites": len(group),
                             "mean MSE": group["site MSE"].mean(), "median MSE": median,
                             "eta": error_ratio(group["baseline MSE"].median(), median)})
display(pd.DataFrame(group_scores))
display(holdout_scores)
for start in range(0, len(panels), 9):
    plot_grid(panels[start:start + 9], "Withheld Zn–I spectra | baseline MSE divided by model MSE")